# 81장 · FastAPI로 구현하는 할 일 CRUD

CRUD는 레코드의 생성·조회·수정·삭제입니다. 서버가 발급한 ID로 같은 할 일을 계속 가리키고, 변경 뒤의 목록을 다시 읽습니다.

브라우저 안에 있던 할 일을 서버에 보냈습니다. 추가 버튼을 눌렀다는 사실만으로 화면 목록을 확정하면 저장에 실패한 항목이 남을 수 있습니다. 실제 FastAPI 경로를 호출해 응답을 받은 뒤에만 화면에 보일 값을 정합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 fastapi 0.135.0, ipython 9.17.1, pydantic 2.14.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## POST와 GET으로 등록·조회

단일 사용자 실습의 저장소가 번호를 발급합니다. 생성 요청은 title만 보내고, 응답 data에는 id·title·completed가 전부 들어옵니다. 아래는 메모리 저장 경로의 축약 구현입니다. 원본에서 관리하는 오류·저장 상태 외피 중 이번 실습은 data에 집중합니다.


### 셀 01 · FastAPI CRUD 경로 등록

POST는 양끝을 다듬은 제목에 새 ID를 배정합니다. PATCH 예제는 완료 변경에 집중하고 DELETE는 삭제한 항목을 돌려줍니다.

노트북 축약판의 라우트부터 읽으세요. 참고 앱의 요청표에는 개별 조회와 제목 변경까지 있지만 축약 라우트에 없는 동작은 여기서 시험할 수 없습니다. 같은 PATCH라도 받는 필드가 completed인지 title인지 확인한 뒤 비교하세요.


In [1]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, StrictBool
from itertools import count

app = FastAPI()
items, ids = {}, count(1)
TRIM = "".join(chr(n) for n in (9, 10, 11, 12, 13, 32, 160, 0x1680, *range(0x2000, 0x200B), 0x2028, 0x2029, 0x202F, 0x205F, 0x3000, 0xFEFF))
class Create(BaseModel):
    model_config = ConfigDict(extra="forbid")
    title: str
class Complete(BaseModel):
    model_config = ConfigDict(extra="forbid")
    completed: StrictBool

def existing(id):
    if id not in items:
        raise HTTPException(404)
    return items[id]

@app.post("/todos", status_code=201)
def create(body: Create):
    title = body.title.strip(TRIM)
    if not title:
        raise HTTPException(422)
    key = next(ids)
    items[key] = {"id": key, "title": title, "completed": False}
    return {"data": items[key]}

@app.get("/todos")
def list_items():
    return {"data": list(items.values())}

@app.patch("/todos/{id}")
def patch(id: int, body: Complete):
    row = existing(id)
    row["completed"] = body.completed
    return {"data": row}

@app.delete("/todos/{id}")
def remove(id: int):
    existing(id)
    return {"data": items.pop(id)}

client = TestClient(app)

경로를 정의했습니다. T1 제목은 JavaScript trim과 같은 25개 공백 문자만 양끝에서 제거합니다. 삭제한 번호를 다시 사용하지 않도록 별도 카운터를 뒀습니다. 개별 조회나 제목 변경을 연습할 때는 참조 앱을 받으세요. 압축을 푼 폴더에서 FastAPI와 uvicorn이 설치된 수업 환경으로 python -m uvicorn app:create_app --factory 명령을 실행하세요. /docs에서 할 일을 등록하고 응답의 ID로 GET /todos/{todo_id}를 연 뒤, PATCH에 {"title":"복습"}을 보내세요. 같은 ID로 다시 조회해 바뀐 제목을 확인합니다.


<p><a download="chapter81_reference.zip" href="">개별 조회와 제목 수정을 지원하는 참조 앱 받기</a></p>


### 셀 02 · POST 응답에서 ID 읽기

POST 응답에서 서버가 정한 번호를 보관합니다.


In [2]:
created = client.post("/todos", json={"title": "  복습  "})
item = created.json()["data"]
print(created.status_code, item)

201 {'id': 1, 'title': '복습', 'completed': False}


201과 함께 id 1, 제목 복습, completed False가 돌아옵니다. 새로고침한 화면은 GET /todos가 반환한 목록으로 다시 그릴 수 있습니다.


## PATCH·DELETE 이후 화면 상태 갱신

완료 버튼을 눌렀을 때 기존 상태를 뒤집으라고 요청하면 두 번 전송했을 때 원래대로 돌아갈 수 있습니다. 원하는 최종 값 true를 보내고 반환된 항목을 화면에 반영합니다.


### 셀 03 · PATCH 후 목록 조회

반환 상태와 목록의 같은 ID가 일치하는지 확인합니다.


In [3]:
response = client.patch(f"/todos/{item['id']}", json={"completed": True})
listed = client.get("/todos").json()["data"][0]
print(response.status_code, listed["id"], listed["completed"])

200 1 True


200 1 True입니다. 제목이 같은 다른 항목이 있어도 요청 경로의 번호가 대상 행을 가리킵니다.


### 셀 04 · 응답 목록의 HTML 렌더링

앞에서 받은 GET의 data를 사용해 노트북 안에 완료 목록을 그립니다. 이어서 없는 항목의 실패 응답을 받아도 직전에 확인한 목록을 유지합니다.


In [4]:
from html import escape
from IPython.display import display

visible = client.get("/todos").json()["data"]
failed = client.patch("/todos/999", json={"completed": True})
if failed.is_success:
    changed = failed.json()["data"]
    visible = [changed if row["id"] == changed["id"] else row for row in visible]
labels = [f"{row['title']} · {'완료' if row['completed'] else '미완료'}" for row in visible]
html = "<ul>" + "".join("<li>" + escape(label) + "</li>" for label in labels) + "</ul>"
display({"text/html": html, "text/plain": " / ".join(labels)}, raw=True)
print(failed.status_code)

복습 · 완료

404


목록에는 ‘복습 · 완료’가 남고 실패한 요청의 상태는 404입니다. 목록은 실제 TestClient 응답으로 그렸습니다. 독립적인 브라우저가 원격 서버를 호출한 실험은 아닙니다.


### 과제 05 · DELETE 결과 검증

기존 실습 항목을 지우고 DELETE 결과 번호와 목록 길이를 출력하세요.


In [5]:
# DELETE /todos/{id}와 GET /todos를 차례로 요청하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>deleted = client.delete(f&quot;/todos/{item[&#x27;id&#x27;]}&quot;)
print(deleted.json()[&#x27;data&#x27;][&#x27;id&#x27;], len(client.get(&#x27;/todos&#x27;).json()[&#x27;data&#x27;]))</code></pre>
<p>실행 결과</p><pre><code>1 0</code></pre>
<p>1 0입니다. 삭제 응답은 지운 레코드를, 재조회 결과는 현재 남은 목록을 보여 줍니다. 실제 DB 재시작 후에도 유지되는지는 다음 장에서 다룹니다.</p>
</details>


## 돌아보기

화면에서 변경을 요청한 시점과 서버 응답으로 확정한 시점을 가리켜 보세요. 응답이 끊겼다면 같은 쓰기를 무작정 반복하기 전에 목록을 조회해 현재 상태를 확인합니다.
